In [ ]:
# =========================
# Colab Script (Drive path + pre-split train/val/test)
# - LightGBM: CPU
# - Train/Val/Test are pre-split CSVs (7:1:2)
# - K_EDGE (step=1)
#   * O    : original features only (fixed)
#   * F    : edge features only (top-k by ranking)            K=1..max_edges
#   * OF   : original + top-k edges (orig always included)    K=1..max_edges
#   * OFR  : RANDOM features from (original ∪ edges)          K in {5,10,20,30,40,"ALL"}
#            - Each K runs N_RANDOM_TRIALS; results_OFR.csv stores MEAN across trials
#            - results_OFR_trials.csv stores every trial
#
# Metrics: F1, AUROC, AUPRC, Brier, ECE
# Outputs: results_O.csv / results_F.csv / results_OF.csv / results_OFR.csv / results_OFR_trials.csv
# =========================

!pip -q install lightgbm xgboost

import os
import random
import warnings
import time
import numpy as np
import pandas as pd

from typing import Dict, List, Tuple, Union, Optional

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from sklearn.metrics import f1_score, roc_auc_score, average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import xgboost as xgb
import lightgbm as lgb

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# -------------------------
# Simple logger
# -------------------------
T0 = time.time()
def log(msg: str):
    dt = time.time() - T0
    print(f"[{dt:8.1f}s] {msg}")

# -------------------------
# Warning / Log control
# -------------------------
warnings.filterwarnings(
    "ignore",
    message=r"X does not have valid feature names, but LGBMClassifier was fitted with feature names.*",
    category=UserWarning,
)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# LightGBM internal logs off
try:
    lgb.register_logger(lambda msg: None)
except Exception:
    pass

# -------------------------
# Config
# -------------------------
RANDOM_STATE = 42
ECE_BINS = 15

# FFMLP config
RUN_FFMLP = True
TORCH_USE_GPU = True

# ✅ deterministic 강제 OFF (CuBLAS 에러 방지)
TORCH_DETERMINISTIC = False

BATCH_SIZE = 2048
EPOCHS = 30
LR = 1e-3
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 5

# F/OF: K_EDGE 1..max_edges (cap optional)
K_EDGE_MAX_CAP: Optional[int] = None

# OFR: K candidates fixed
OFR_K_CANDIDATES: List[Union[int, str]] = [5, 10, 20, 30, 40, "ALL"]
N_RANDOM_TRIALS = 5  # K마다 N회

# 진행 로그 빈도 (F/OF)
LOG_EVERY_K = 10  # 1이면 매 K마다 출력

# LightGBM CPU params
LGBM_CPU_PARAMS = dict(
    n_estimators=2000,
    learning_rate=0.02,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)

BASE_DIR = "/content/drive/MyDrive/bank_failure_prediction"
OUT_DIR = os.path.join(BASE_DIR, "results_tables")
os.makedirs(OUT_DIR, exist_ok=True)

DATA_BASE = {
    "train": os.path.join(BASE_DIR, "data_with_features_train.csv"),
    "val":   os.path.join(BASE_DIR, "data_with_features_val.csv"),
    "test":  os.path.join(BASE_DIR, "data_with_features_test.csv"),
}

DATA_DAG = {
    "NOTEARS": {
        "train": os.path.join(BASE_DIR, "data_with_features_NOTEARS_train.csv"),
        "val":   os.path.join(BASE_DIR, "data_with_features_NOTEARS_val.csv"),
        "test":  os.path.join(BASE_DIR, "data_with_features_NOTEARS_test.csv"),
    },
    "PC": {
        "train": os.path.join(BASE_DIR, "data_with_features_PC_train.csv"),
        "val":   os.path.join(BASE_DIR, "data_with_features_PC_val.csv"),
        "test":  os.path.join(BASE_DIR, "data_with_features_PC_test.csv"),
    },
    "GES": {
        "train": os.path.join(BASE_DIR, "data_with_features_GES_train.csv"),
        "val":   os.path.join(BASE_DIR, "data_with_features_GES_val.csv"),
        "test":  os.path.join(BASE_DIR, "data_with_features_GES_test.csv"),
    },
    "GOLEM": {
        "train": os.path.join(BASE_DIR, "data_with_features_GOLEM_train.csv"),
        "val":   os.path.join(BASE_DIR, "data_with_features_GOLEM_val.csv"),
        "test":  os.path.join(BASE_DIR, "data_with_features_GOLEM_test.csv"),
    },
}

# -------------------------
# Seed control
# -------------------------
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = False  # deterministic OFF

    if TORCH_DETERMINISTIC:
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass
    else:
        try:
            torch.use_deterministic_algorithms(False)
        except Exception:
            pass

set_seed(RANDOM_STATE)
log(f"seed set: {RANDOM_STATE} | torch cuda={torch.cuda.is_available()} | RUN_FFMLP={RUN_FFMLP}")

# -------------------------
# Metrics
# -------------------------
def expected_calibration_error(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 15) -> float:
    y_true = y_true.astype(int)
    y_prob = np.clip(y_prob, 0.0, 1.0)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    n = len(y_true)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (y_prob >= lo) & (y_prob < hi) if i < n_bins - 1 else (y_prob >= lo) & (y_prob <= hi)
        if not np.any(mask):
            continue
        acc = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += (mask.sum() / n) * abs(acc - conf)
    return float(ece)

def brier_score(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    y_true = y_true.astype(float)
    y_prob = np.clip(y_prob, 0.0, 1.0)
    return float(np.mean((y_prob - y_true) ** 2))

def best_f1_threshold(y_true: np.ndarray, y_prob: np.ndarray, n_grid: int = 101) -> float:
    thresholds = np.linspace(0.0, 1.0, n_grid)
    best_t, best_f1 = 0.5, -1.0
    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t)

def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float) -> Dict[str, float]:
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "AUROC": float(roc_auc_score(y_true, y_prob)),
        "AUPRC": float(average_precision_score(y_true, y_prob)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "Brier": brier_score(y_true, y_prob),
        "ECE": expected_calibration_error(y_true, y_prob, n_bins=ECE_BINS),
    }

# -------------------------
# Data utils
# -------------------------
def detect_target_col(df: pd.DataFrame) -> str:
    candidates = ["label", "target", "y", "failure", "bank_failure", "default", "is_failed"]
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"Target column not found. Tried {candidates}")

def is_index_col(c: str) -> bool:
    return c.startswith("Unnamed") or c.lower() in {"index", "_index"}

def get_original_cols(df_any: pd.DataFrame, target_col: str) -> List[str]:
    return [c for c in df_any.columns
            if c != target_col and (not c.startswith("edge_")) and (not is_index_col(c))]

def ensure_cols(df: pd.DataFrame, cols: List[str], name: str):
    miss = [c for c in cols if c not in df.columns]
    if miss:
        raise ValueError(f"[{name}] Missing columns: {miss[:30]} (total {len(miss)})")

def to_numpy32(df: pd.DataFrame, cols: List[str]) -> np.ndarray:
    return df[cols].to_numpy(dtype=np.float32)

# -------------------------
# Edge ranking by train std
# -------------------------
def get_edge_cols(df_alg: pd.DataFrame, alg: str) -> List[str]:
    prefix = f"edge_{alg}__"
    cols = [c for c in df_alg.columns if c.startswith(prefix)]
    if len(cols) == 0:
        cols = [c for c in df_alg.columns if c.startswith("edge_")]
    return cols

def rank_edges_by_train_std(df_alg_train: pd.DataFrame, edge_cols: List[str]) -> List[str]:
    if len(edge_cols) == 0:
        return []
    stds = (
        df_alg_train[edge_cols]
        .astype(float)
        .std(axis=0)
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )
    return stds.sort_values(ascending=False).index.tolist()

# -------------------------
# Random feature picker (OFR)
# -------------------------
def pick_random_cols(pool_cols: List[str], n_pick: int, seed: int) -> List[str]:
    rng = np.random.default_rng(seed)
    n_pick = min(int(n_pick), len(pool_cols))
    idx = rng.choice(len(pool_cols), size=n_pick, replace=False)
    return [pool_cols[i] for i in idx]

# -------------------------
# Models
# -------------------------
def fit_predict_logit(X_train, y_train, X_val, y_val, X_test):
    clf = LogisticRegression(max_iter=2000, solver="lbfgs")
    clf.fit(X_train, y_train)
    val_prob = clf.predict_proba(X_val)[:, 1]
    thr = best_f1_threshold(y_val, val_prob)
    test_prob = clf.predict_proba(X_test)[:, 1]
    return test_prob, thr

def fit_predict_rf(X_train, y_train, X_val, y_val, X_test):
    clf = RandomForestClassifier(
        n_estimators=500,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced_subsample",
    )
    clf.fit(X_train, y_train)
    val_prob = clf.predict_proba(X_val)[:, 1]
    thr = best_f1_threshold(y_val, val_prob)
    test_prob = clf.predict_proba(X_test)[:, 1]
    return test_prob, thr

def fit_predict_xgb(X_train, y_train, X_val, y_val, X_test):
    params_gpu = dict(
        tree_method="hist",
        device="cuda",
        n_estimators=800,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
    )
    params_cpu = dict(params_gpu)
    params_cpu["device"] = "cpu"

    try:
        clf = xgb.XGBClassifier(**params_gpu)
        clf.fit(X_train, y_train)
        used = "GPU"
    except Exception:
        clf = xgb.XGBClassifier(**params_cpu)
        clf.fit(X_train, y_train)
        used = "CPU"

    val_prob = clf.predict_proba(X_val)[:, 1]
    thr = best_f1_threshold(y_val, val_prob)
    test_prob = clf.predict_proba(X_test)[:, 1]
    return test_prob, thr, used

def fit_predict_lgbm_cpu(X_train, y_train, X_val, y_val, X_test):
    clf = lgb.LGBMClassifier(**LGBM_CPU_PARAMS)
    clf.fit(X_train, y_train)
    val_prob = clf.predict_proba(X_val)[:, 1]
    thr = best_f1_threshold(y_val, val_prob)
    test_prob = clf.predict_proba(X_test)[:, 1]
    return test_prob, thr

# -------------------------
# FFMLP
# -------------------------
class FFMLP(nn.Module):
    def __init__(self, in_dim: int, hidden: List[int] = [128, 64], dropout: float = 0.1):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)

@torch.no_grad()
def predict_proba_mlp(model: nn.Module, X: np.ndarray, device: str) -> np.ndarray:
    model.eval()
    dl = DataLoader(TensorDataset(torch.tensor(X)), batch_size=4096, shuffle=False)
    probs = []
    for (xb,) in dl:
        xb = xb.to(device)
        logits = model(xb)
        p = torch.sigmoid(logits).detach().cpu().numpy()
        probs.append(p)
    return np.concatenate(probs, axis=0)

def train_mlp(X_train, y_train, X_val, y_val, seed: int) -> Tuple[FFMLP, float, str]:
    set_seed(seed)
    device = "cuda" if (TORCH_USE_GPU and torch.cuda.is_available()) else "cpu"

    model = FFMLP(in_dim=X_train.shape[1]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()

    train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    val_ds = TensorDataset(torch.tensor(X_val), torch.tensor(y_val))

    gen = torch.Generator()
    gen.manual_seed(seed)

    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=gen)
    val_dl = DataLoader(val_ds, batch_size=4096, shuffle=False)

    best_val = float("inf")
    best_state = None
    bad = 0

    for ep in range(1, EPOCHS + 1):
        model.train()
        for xb, yb in train_dl:
            xb = xb.to(device)
            yb = yb.to(device)
            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()

        model.eval()
        vlosses = []
        with torch.no_grad():
            for xb, yb in val_dl:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                vlosses.append(loss_fn(logits, yb).item())
        v = float(np.mean(vlosses))

        if v < best_val - 1e-6:
            best_val = v
            best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= EARLY_STOPPING_PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    y_val_prob = predict_proba_mlp(model, X_val, device=device)
    thr = best_f1_threshold(y_val, y_val_prob)
    return model, thr, device

# -------------------------
# Evaluation runner
# -------------------------
def eval_all_models_fixed(
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    seed: int,
    run_ffmlp: bool = True
) -> List[Dict[str, float]]:
    out = []

    t0 = time.time()
    prob, thr = fit_predict_lgbm_cpu(X_train, y_train, X_val, y_val, X_test)
    out.append({"MODEL": "LightGBM", **compute_metrics(y_test, prob, thr)})
    t_lgb = time.time() - t0

    t0 = time.time()
    prob, thr, used = fit_predict_xgb(X_train, y_train, X_val, y_val, X_test)
    out.append({"MODEL": f"XGBoost({used})", **compute_metrics(y_test, prob, thr)})
    t_xgb = time.time() - t0

    if run_ffmlp:
        t0 = time.time()
        model_mlp, thr, dev = train_mlp(
            X_train.astype(np.float32), y_train.astype(np.float32),
            X_val.astype(np.float32),   y_val.astype(np.float32),
            seed=seed
        )
        prob = predict_proba_mlp(model_mlp, X_test.astype(np.float32), device=dev)
        out.append({"MODEL": f"FFMLP({dev})", **compute_metrics(y_test, prob, thr)})
        t_mlp = time.time() - t0
    else:
        t_mlp = 0.0

    t0 = time.time()
    prob, thr = fit_predict_logit(X_train, y_train, X_val, y_val, X_test)
    out.append({"MODEL": "Logit", **compute_metrics(y_test, prob, thr)})
    t_logit = time.time() - t0

    t0 = time.time()
    prob, thr = fit_predict_rf(X_train, y_train, X_val, y_val, X_test)
    out.append({"MODEL": "RF", **compute_metrics(y_test, prob, thr)})
    t_rf = time.time() - t0

    # timings included for progress visibility (not saved)
    log(f"  models done | LGB={t_lgb:.1f}s XGB={t_xgb:.1f}s MLP={t_mlp:.1f}s LOGIT={t_logit:.1f}s RF={t_rf:.1f}s")
    return out

# -------------------------
# Load base once
# -------------------------
log("checking base files...")
for k in ["train", "val", "test"]:
    if not os.path.exists(DATA_BASE[k]):
        raise FileNotFoundError(f"Missing base {k}: {DATA_BASE[k]}")
log("loading base csvs...")
df_base_train = pd.read_csv(DATA_BASE["train"], low_memory=False)
df_base_val   = pd.read_csv(DATA_BASE["val"],   low_memory=False)
df_base_test  = pd.read_csv(DATA_BASE["test"],  low_memory=False)

target_col = detect_target_col(df_base_train)
orig_cols = get_original_cols(df_base_train, target_col)

ensure_cols(df_base_val,  orig_cols + [target_col], "BASE_VAL")
ensure_cols(df_base_test, orig_cols + [target_col], "BASE_TEST")

X_train_base = to_numpy32(df_base_train, orig_cols)
X_val_base   = to_numpy32(df_base_val,   orig_cols)
X_test_base  = to_numpy32(df_base_test,  orig_cols)

y_train_base = df_base_train[target_col].to_numpy(dtype=np.int64)
y_val_base   = df_base_val[target_col].to_numpy(dtype=np.int64)
y_test_base  = df_base_test[target_col].to_numpy(dtype=np.int64)

log(f"base loaded | n_train={len(df_base_train)} n_val={len(df_base_val)} n_test={len(df_base_test)}")
log(f"target_col={target_col} | n_orig_cols={len(orig_cols)}")

# -------------------------
# Run
# -------------------------
rows_O, rows_F, rows_OF, rows_OFR = [], [], [], []
rows_OFR_trials = []

log("[RUN] O (once)")
t0 = time.time()
o_metrics_once = eval_all_models_fixed(
    X_train_base, y_train_base,
    X_val_base,   y_val_base,
    X_test_base,  y_test_base,
    seed=RANDOM_STATE,
    run_ffmlp=RUN_FFMLP
)
log(f"O (once) done in {time.time() - t0:.1f}s")

for alg in DATA_DAG.keys():
    for m in o_metrics_once:
        rows_O.append({"SET": "O", "DAG": alg, "K_EDGE": 0, **m})

for alg, paths in DATA_DAG.items():
    log(f"[ALG] {alg} loading...")
    for k in ["train", "val", "test"]:
        if not os.path.exists(paths[k]):
            raise FileNotFoundError(f"Missing {alg} {k}: {paths[k]}")

    df_alg_train = pd.read_csv(paths["train"], low_memory=False)
    df_alg_val   = pd.read_csv(paths["val"],   low_memory=False)
    df_alg_test  = pd.read_csv(paths["test"],  low_memory=False)

    if target_col not in df_alg_train.columns or target_col not in df_alg_val.columns or target_col not in df_alg_test.columns:
        raise ValueError(f"[{alg}] target_col '{target_col}' not found in train/val/test")

    y_train_alg = df_alg_train[target_col].to_numpy(dtype=np.int64)
    y_val_alg   = df_alg_val[target_col].to_numpy(dtype=np.int64)
    y_test_alg  = df_alg_test[target_col].to_numpy(dtype=np.int64)

    ensure_cols(df_alg_train, orig_cols + [target_col], f"{alg}_TRAIN_ORIG")
    ensure_cols(df_alg_val,   orig_cols + [target_col], f"{alg}_VAL_ORIG")
    ensure_cols(df_alg_test,  orig_cols + [target_col], f"{alg}_TEST_ORIG")

    edge_cols_all = get_edge_cols(df_alg_train, alg)
    edge_ranked_cols = rank_edges_by_train_std(df_alg_train, edge_cols_all)

    max_edges = len(edge_ranked_cols)
    if K_EDGE_MAX_CAP is not None:
        max_edges = min(max_edges, int(K_EDGE_MAX_CAP))

    log(f"[ALG] {alg} ready | edges(total)={len(edge_ranked_cols)} | run F/OF K=1..{max_edges}")

    # ---- F / OF ----
    for k_edge in range(1, max_edges + 1):
        edge_cols_used = edge_ranked_cols[:k_edge]

        seed_run = RANDOM_STATE + (abs(hash(alg)) % 100000) * 1000 + k_edge

        # F
        X_train_F = to_numpy32(df_alg_train, edge_cols_used)
        X_val_F   = to_numpy32(df_alg_val,   edge_cols_used)
        X_test_F  = to_numpy32(df_alg_test,  edge_cols_used)

        if (k_edge == 1) or (k_edge == max_edges) or (k_edge % LOG_EVERY_K == 0):
            log(f"  [F ] {alg} K={k_edge}/{max_edges} (n_feat={X_train_F.shape[1]})")

        f_metrics = eval_all_models_fixed(
            X_train_F, y_train_alg,
            X_val_F,   y_val_alg,
            X_test_F,  y_test_alg,
            seed=seed_run,
            run_ffmlp=RUN_FFMLP
        )
        for m in f_metrics:
            rows_F.append({"SET": "F", "DAG": alg, "K_EDGE": k_edge, **m})

        # OF
        of_cols = orig_cols + edge_cols_used
        X_train_OF = to_numpy32(df_alg_train, of_cols)
        X_val_OF   = to_numpy32(df_alg_val,   of_cols)
        X_test_OF  = to_numpy32(df_alg_test,  of_cols)

        if (k_edge == 1) or (k_edge == max_edges) or (k_edge % LOG_EVERY_K == 0):
            log(f"  [OF] {alg} K={k_edge}/{max_edges} (n_feat={X_train_OF.shape[1]})")

        of_metrics = eval_all_models_fixed(
            X_train_OF, y_train_alg,
            X_val_OF,   y_val_alg,
            X_test_OF,  y_test_alg,
            seed=seed_run,
            run_ffmlp=RUN_FFMLP
        )
        for m in of_metrics:
            rows_OF.append({"SET": "OF", "DAG": alg, "K_EDGE": k_edge, **m})

    # ---- OFR ----
    pool_cols = orig_cols + edge_ranked_cols
    pool_size = len(pool_cols)

    def resolve_k(k: Union[int, str]) -> int:
        if k == "ALL":
            return pool_size
        return min(int(k), pool_size)

    log(f"[OFR] {alg} pool_size={pool_size} | K={OFR_K_CANDIDATES} | trials={N_RANDOM_TRIALS}")

    for k_raw in OFR_K_CANDIDATES:
        k_pick = resolve_k(k_raw)
        if k_pick <= 0:
            continue

        log(f"  [OFR] {alg} K={k_raw} -> pick={k_pick} starting...")

        trial_rows = []
        for t in range(N_RANDOM_TRIALS):
            seed_trial = (
                RANDOM_STATE
                + (abs(hash(alg)) % 100000) * 100000
                + (k_pick * 1000)
                + t
            )

            ofr_cols = pick_random_cols(pool_cols, n_pick=k_pick, seed=seed_trial)
            X_train_OFR = to_numpy32(df_alg_train, ofr_cols)
            X_val_OFR   = to_numpy32(df_alg_val,   ofr_cols)
            X_test_OFR  = to_numpy32(df_alg_test,  ofr_cols)

            log(f"    trial {t+1}/{N_RANDOM_TRIALS} seed={seed_trial} n_feat={X_train_OFR.shape[1]}")
            ofr_metrics = eval_all_models_fixed(
                X_train_OFR, y_train_alg,
                X_val_OFR,   y_val_alg,
                X_test_OFR,  y_test_alg,
                seed=seed_trial,
                run_ffmlp=RUN_FFMLP
            )

            for m in ofr_metrics:
                row = {
                    "SET": "OFR",
                    "DAG": alg,
                    "MODEL": m["MODEL"],
                    "K_EDGE": ("ALL" if k_raw == "ALL" else int(k_raw)),
                    "TRIAL": t,
                    "SEED": seed_trial,
                    "AUROC": m["AUROC"],
                    "AUPRC": m["AUPRC"],
                    "F1": m["F1"],
                    "Brier": m["Brier"],
                    "ECE": m["ECE"],
                }
                trial_rows.append(row)
                rows_OFR_trials.append(row)

        df_trials = pd.DataFrame(trial_rows)
        for model_name, g in df_trials.groupby("MODEL"):
            rows_OFR.append({
                "SET": "OFR",
                "DAG": alg,
                "MODEL": model_name,
                "K_EDGE": ("ALL" if k_raw == "ALL" else int(k_raw)),
                "AUROC": float(g["AUROC"].mean()),
                "AUPRC": float(g["AUPRC"].mean()),
                "F1": float(g["F1"].mean()),
                "Brier": float(g["Brier"].mean()),
                "ECE": float(g["ECE"].mean()),
            })

        log(f"  [OFR] {alg} K={k_raw} done (mean aggregated)")

    log(f"[ALG] {alg} done")

# -------------------------
# Save CSVs
# -------------------------
def finalize_table(rows: List[dict], set_name: str) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    if df.empty:
        return pd.DataFrame(columns=["SET","DAG","MODEL","K_EDGE","AUROC","AUPRC","F1","Brier","ECE"])
    df = df[["SET", "DAG", "MODEL", "K_EDGE", "AUROC", "AUPRC", "F1", "Brier", "ECE"]]
    df = df.sort_values(
        ["DAG", "K_EDGE", "AUROC", "AUPRC", "MODEL"],
        ascending=[True, True, False, False, True]
    ).reset_index(drop=True)
    df["SET"] = set_name
    return df

log("finalizing tables...")
tbl_O   = finalize_table(rows_O,   "O")
tbl_F   = finalize_table(rows_F,   "F")
tbl_OF  = finalize_table(rows_OF,  "OF")
tbl_OFR = finalize_table(rows_OFR, "OFR")

out_o   = os.path.join(OUT_DIR, "results_O.csv")
out_f   = os.path.join(OUT_DIR, "results_F.csv")
out_of  = os.path.join(OUT_DIR, "results_OF.csv")
out_ofr = os.path.join(OUT_DIR, "results_OFR.csv")
out_ofr_trials = os.path.join(OUT_DIR, "results_OFR_trials.csv")

tbl_O.to_csv(out_o, index=False)
tbl_F.to_csv(out_f, index=False)
tbl_OF.to_csv(out_of, index=False)
tbl_OFR.to_csv(out_ofr, index=False)
pd.DataFrame(rows_OFR_trials).to_csv(out_ofr_trials, index=False)

log("[DONE] saved:")
log(out_o)
log(out_f)
log(out_of)
log(out_ofr)
log(out_ofr_trials)

display(tbl_O.head(10))
display(tbl_F.head(10))
display(tbl_OF.head(10))
display(tbl_OFR.head(10))
